<center><h1>Companhia Aberta Demonstrativo Financeiro</h1></center>

# <h2>Load Libraries</h2>

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 10)

# <h2>Constants</h2>

In [7]:
CD_CONTA = 'CD_CONTA'
DS_CONTA = 'DS_CONTA'

In [8]:
filename = 'dfp_cia_aberta_BPA_con_2020.csv'

In [9]:
CNPJ_CIA = '97.837.181/0001-47'
ORDEM_EXERC = 'ÚLTIMO'

# <h2>Import Data</h2>

## <h3>Load .csv</h3>

In [10]:
try:
    cia_aberta_df = pd.read_csv(filename, encoding='ISO-8859-1', sep=";")
except Exception as e:
    print(f"Error: {e}")

## <h3>Select Company</h3>

In [11]:
df = cia_aberta_df[cia_aberta_df['CNPJ_CIA'] == CNPJ_CIA].copy()    # seleciono somente as linhas relativas a uma companhia de interesse
df = df[df['ORDEM_EXERC'] == ORDEM_EXERC]                                         # seleciono somente o último ou penúltimo exercício
df = df[[CD_CONTA, DS_CONTA,'VL_CONTA']]
df.reset_index(inplace=True, drop=True)

In [12]:
df

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0
1,1.01,Ativo Circulante,4220022.0
2,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Aplicações Financeiras,0.0
4,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...
71,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


# <h2>Wrangling</h2>

## <h3>Number of Steps</h3>

In [13]:
"""
CD_CONTA is a column which contains codes. Each code was generated by chaining numbers and dots, like: 1, 1.01, 1.01.01, 1.01.02...
Each new dot in front of a number represents a sub hierarchy. For example: 1.01 is a subset of 1, 1.01.01 is a subset of 1.01, etc.
We want to find which is the maximum code lenght, which corrisponds to the total number of levels (ranks) in the hierarchy.
First, we need to split each code as a list of numbers:  1.01 thus becomes [1, 01], 1.01.01 becomes [1, 01, 01], etc.
We store this column of lists in a variable called cd_conta_split.
Then, we calculate the lengths of each one of these lists and store the result in a vaiable called cd_conta_len.
Finally, we find the number of ranks using max()
"""
cd_conta_split = df[CD_CONTA].str.split('.')
cd_conta_len = [len(lst) for lst in cd_conta_split]    # len(row) or len(lst)?
num_steps = max(cd_conta_len)

In [14]:
num_steps

5

## <h3>ROUND 1</h3>

In [15]:
round=1

In [16]:
df

,CD_CONTA,DS_CONTA,VL_CONTA
0,1,Ativo Total,11498520.0
1,1.01,Ativo Circulante,4220022.0
2,1.01.01,Caixa e Equivalentes de Caixa,1728413.0
3,1.01.02,Aplicações Financeiras,0.0
4,1.01.02.01,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...
71,1.02.04.02.07,Goodwill na aquisição da Caetex Florestal,8767.0
72,1.02.04.02.08,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,1.02.04.02.09,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,1.02.04.02.10,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Join</h4>

In [20]:
"""
CD_CONTA is a column which contains codes. Each code was generated by chaining numbers and dots, like: 1, 1.01, 1.01.01, 1.01.02...
Each new dot in front of a number represents a sub hierarchy. For example: 1.01 is a subset of 1, 1.01.01 is a subset of 1.01, etc.
We want to find which codes have minimum lenght. The minimum length should correspond to the round number (the minimum length at round 1 should be 1, at round 2 should be 2, etc.).
First, we need to split each code as a list of numbers:  1.01 thus becomes [1, 01], 1.01.01 becomes [1, 01, 01], etc.
We store this column of lists in a variable called cd_conta_split.
Then, we calculate the lengths of each one of these lists and store the result in a vaiable called cd_conta_len.
Finally, we find the number of ranks using max()
"""
cd_conta_split = df[CD_CONTA].str.split('.')
cd_conta_joincol = [row[:round] for row in cd_conta_split]    # row[:1] round = 1

Select rows with minimum length

In [ ]:
len_key = [len(row) for row in cd_conta_split]        # len(row) or len(lst)?
min_len = np.array(len_key).min()    # min_len = minimum length
ind_min_len = np.where(len_key==min_len)[0]    # ind_min_len = index of key 'CD_CONTA' with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA
0,ÚLTIMO,1,Ativo Total,11498520.0


In [ ]:
keys = [cd_conta_joincol[ind] for ind in ind_min_len]
key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

Join:

In [ ]:
ds_conta_1 = [df['DS_CONTA'][key_idx_dict[key]]
              for key in key_idx_dict
              for row in cd_conta_joincol
              if ".".join(row) == key]

In [ ]:
ds_conta_1 = [df['DS_CONTA'][key_idx_dict[key]]
              for row in cd_conta_split
              for key in key_idx_dict
              if ".".join(row[:1]) == key]

Insert:

In [ ]:
df.insert(2, column='DS_CONTA_1', value=ds_conta_1)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,ÚLTIMO,1,Ativo Total,Ativo Total,11498520.0
1,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,4220022.0
2,ÚLTIMO,1.01.01,Ativo Total,Caixa e Equivalentes de Caixa,1728413.0
3,ÚLTIMO,1.01.02,Ativo Total,Aplicações Financeiras,0.0
4,ÚLTIMO,1.01.02.01,Ativo Total,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...,...
71,ÚLTIMO,1.02.04.02.07,Ativo Total,Goodwill na aquisição da Caetex Florestal,8767.0
72,ÚLTIMO,1.02.04.02.08,Ativo Total,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
73,ÚLTIMO,1.02.04.02.09,Ativo Total,Goodwill na aquisição da Massima Revestimentos...,6110.0
74,ÚLTIMO,1.02.04.02.10,Ativo Total,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Drop rows</h4>

Select rows with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,ÚLTIMO,1,Ativo Total,Ativo Total,11498520.0


For each key which was used to join, check if there is at least another row in `CD_CONTA` which code starts with the same key (example: `CD_CONTA` '1.01' starting with '1'; `CD_CONTA` '1.01.01' starting with '1.01', etc.)

In [ ]:
for ind in ind_min_len:
    mask = [True]*len(ds_conta_1)
    mask[ind] = False
    if any(row[:min_len] == cd_conta_joincol[ind] for row in cd_conta_split[mask]):
        df = df.drop(ind)
    else:
        df.loc[ind, 'CD_CONTA'] = df.loc[ind, 'CD_CONTA'] + '.00'

df.reset_index(inplace=True, drop=True)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,4220022.0
1,ÚLTIMO,1.01.01,Ativo Total,Caixa e Equivalentes de Caixa,1728413.0
2,ÚLTIMO,1.01.02,Ativo Total,Aplicações Financeiras,0.0
3,ÚLTIMO,1.01.02.01,Ativo Total,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.01.01,Ativo Total,Títulos para Negociação,0.0
...,...,...,...,...,...
70,ÚLTIMO,1.02.04.02.07,Ativo Total,Goodwill na aquisição da Caetex Florestal,8767.0
71,ÚLTIMO,1.02.04.02.08,Ativo Total,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
72,ÚLTIMO,1.02.04.02.09,Ativo Total,Goodwill na aquisição da Massima Revestimentos...,6110.0
73,ÚLTIMO,1.02.04.02.10,Ativo Total,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


## <h3>ROUND 2</h3>

In [ ]:
round=2

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,4220022.0
1,ÚLTIMO,1.01.01,Ativo Total,Caixa e Equivalentes de Caixa,1728413.0
2,ÚLTIMO,1.01.02,Ativo Total,Aplicações Financeiras,0.0
3,ÚLTIMO,1.01.02.01,Ativo Total,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.01.01,Ativo Total,Títulos para Negociação,0.0
...,...,...,...,...,...
70,ÚLTIMO,1.02.04.02.07,Ativo Total,Goodwill na aquisição da Caetex Florestal,8767.0
71,ÚLTIMO,1.02.04.02.08,Ativo Total,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
72,ÚLTIMO,1.02.04.02.09,Ativo Total,Goodwill na aquisição da Massima Revestimentos...,6110.0
73,ÚLTIMO,1.02.04.02.10,Ativo Total,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Join</h4>

In [ ]:
cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
cd_conta_joincol = [row[:2] for row in cd_conta_split]
# cd_conta_joincol: column with keys used to join
# row[:2] round = 2

Select rows with minimum length

In [ ]:
len_key = [len(row) for row in cd_conta_split]  # len(row) = number of elements separated by '.' in CD_CONTA
min_len = np.array(len_key).min()  # min_len = minimum length
ind_min_len = np.where(len_key==min_len)[0]  # ind_min_len = index of key 'CD_CONTA' with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,4220022.0
23,ÚLTIMO,1.02,Ativo Total,Ativo Não Circulante,7278498.0


In [ ]:
keys = [cd_conta_joincol[ind] for ind in ind_min_len]
key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

Join:

In [ ]:
ds_conta_2 = [df['DS_CONTA'][key_idx_dict[key]]
              for key in key_idx_dict
              for row in cd_conta_joincol
              if ".".join(row) == key]

In [ ]:
ds_conta_2 = [df['DS_CONTA'][key_idx_dict[key]]
              for row in cd_conta_split
              for key in key_idx_dict
              if ".".join(row[:2]) == key]

Insert:

In [ ]:
df.insert(3, column='DS_CONTA_2', value=ds_conta_2)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,Ativo Circulante,4220022.0
1,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
2,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
3,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Títulos para Negociação,0.0
...,...,...,...,...,...,...
70,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Caetex Florestal,8767.0
71,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
72,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Massima Revestimentos...,6110.0
73,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Drop rows</h4>

Select rows with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01,Ativo Total,Ativo Circulante,Ativo Circulante,4220022.0
23,ÚLTIMO,1.02,Ativo Total,Ativo Não Circulante,Ativo Não Circulante,7278498.0


For each key which was used to join, check if there is at least another row in `CD_CONTA` which code starts with the same key (example: `CD_CONTA` '1.01' starting with '1'; `CD_CONTA` '1.01.01' starting with '1.01', etc.)

In [ ]:
for ind in ind_min_len:
    mask = [True]*len(ds_conta_2)
    mask[ind] = False
    if any(row[:min_len] == cd_conta_joincol[ind] for row in cd_conta_split[mask]):
        df = df.drop(ind)
    else:
        df.loc[ind, 'CD_CONTA'] = df.loc[ind, 'CD_CONTA'] + '.00'

df.reset_index(inplace=True, drop=True)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
2,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Títulos para Negociação,0.0
4,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Títulos Designados a Valor Justo,0.0
...,...,...,...,...,...,...
68,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Caetex Florestal,8767.0
69,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
70,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Massima Revestimentos...,6110.0
71,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


## <h3>ROUND 3</h3>

In [ ]:
round=3

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
2,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Títulos para Negociação,0.0
4,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Títulos Designados a Valor Justo,0.0
...,...,...,...,...,...,...
68,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Caetex Florestal,8767.0
69,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
70,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Massima Revestimentos...,6110.0
71,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Join</h4>

In [ ]:
cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
cd_conta_joincol = [row[:3] for row in cd_conta_split]
# cd_conta_joincol: column with keys used to join
# row[:3] round = 3

Select rows with minimum length

In [ ]:
len_key = [len(row) for row in cd_conta_split]  # len(row) = number of elements separated by '.' in CD_CONTA
min_len = np.array(len_key).min()  # min_len = minimum length
ind_min_len = np.where(len_key==min_len)[0]  # ind_min_len = index of key 'CD_CONTA' with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,0.0
7,ÚLTIMO,1.01.03,Ativo Total,Ativo Circulante,Contas a Receber,1318743.0
13,ÚLTIMO,1.01.04,Ativo Total,Ativo Circulante,Estoques,924743.0
14,ÚLTIMO,1.01.05,Ativo Total,Ativo Circulante,Ativos Biológicos,0.0
...,...,...,...,...,...,...
18,ÚLTIMO,1.01.08,Ativo Total,Ativo Circulante,Outros Ativos Circulantes,71667.0
22,ÚLTIMO,1.02.01,Ativo Total,Ativo Não Circulante,Ativo Realizável a Longo Prazo,2071636.0
46,ÚLTIMO,1.02.02,Ativo Total,Ativo Não Circulante,Investimentos,963437.0
52,ÚLTIMO,1.02.03,Ativo Total,Ativo Não Circulante,Imobilizado,3512641.0


In [ ]:
keys = [cd_conta_joincol[ind] for ind in ind_min_len]
key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

Join:

In [ ]:
ds_conta_3 = [df['DS_CONTA'][key_idx_dict[key]]
              for key in key_idx_dict
              for row in cd_conta_joincol
              if ".".join(row) == key]

In [ ]:
ds_conta_3 = [df['DS_CONTA'][key_idx_dict[key]]
              for row in cd_conta_split
              for key in key_idx_dict
              if ".".join(row[:3]) == key]

Insert:

In [ ]:
df.insert(4, column='DS_CONTA_3', value=ds_conta_3)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras,0.0
2,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
3,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
4,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
...,...,...,...,...,...,...,...
68,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
69,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
70,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
71,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Drop rows</h4>

Select rows with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras,0.0
7,ÚLTIMO,1.01.03,Ativo Total,Ativo Circulante,Contas a Receber,Contas a Receber,1318743.0
13,ÚLTIMO,1.01.04,Ativo Total,Ativo Circulante,Estoques,Estoques,924743.0
14,ÚLTIMO,1.01.05,Ativo Total,Ativo Circulante,Ativos Biológicos,Ativos Biológicos,0.0
...,...,...,...,...,...,...,...
18,ÚLTIMO,1.01.08,Ativo Total,Ativo Circulante,Outros Ativos Circulantes,Outros Ativos Circulantes,71667.0
22,ÚLTIMO,1.02.01,Ativo Total,Ativo Não Circulante,Ativo Realizável a Longo Prazo,Ativo Realizável a Longo Prazo,2071636.0
46,ÚLTIMO,1.02.02,Ativo Total,Ativo Não Circulante,Investimentos,Investimentos,963437.0
52,ÚLTIMO,1.02.03,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado,3512641.0


For each key which was used to join, check if there is at least another row in `CD_CONTA` which code starts with the same key (example: `CD_CONTA` '1.01' starting with '1'; `CD_CONTA` '1.01.01' starting with '1.01', etc.)

In [ ]:
for ind in ind_min_len:
    mask = [True]*len(ds_conta_3)
    mask[ind] = False
    if any(row[:min_len] == cd_conta_joincol[ind] for row in cd_conta_split[mask]):
        df = df.drop(ind)
    else:
        df.loc[ind, 'CD_CONTA'] = df.loc[ind, 'CD_CONTA'] + '.00'

df.reset_index(inplace=True, drop=True)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
2,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
3,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
4,ÚLTIMO,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...,...,...,...
60,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
61,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
62,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
63,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


## <h3>ROUND 4</h3>

In [ ]:
round=4

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
2,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos para Negociação,0.0
3,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Títulos Designados a Valor Justo,0.0
4,ÚLTIMO,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...,...,...,...
60,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Caetex Florestal,8767.0
61,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
62,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Massima Revestimentos...,6110.0
63,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Join</h4>

In [ ]:
cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
cd_conta_joincol = [row[:4] for row in cd_conta_split]
# cd_conta_joincol: column with keys used to join
# row[:4] round = 4

Select rows with minimum length

In [ ]:
len_key = [len(row) for row in cd_conta_split]  # len(row) = number of elements separated by '.' in CD_CONTA
min_len = np.array(len_key).min()  # min_len = minimum length
ind_min_len = np.where(len_key==min_len)[0]  # ind_min_len = index of key 'CD_CONTA' with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
5,ÚLTIMO,1.01.02.03,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
6,ÚLTIMO,1.01.03.01,Ativo Total,Ativo Circulante,Contas a Receber,Clientes,1239315.0
...,...,...,...,...,...,...,...
46,ÚLTIMO,1.02.03.01,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Operação,3377237.0
47,ÚLTIMO,1.02.03.02,Ativo Total,Ativo Não Circulante,Imobilizado,Direito de Uso em Arrendamento,0.0
48,ÚLTIMO,1.02.03.03,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Andamento,135404.0
49,ÚLTIMO,1.02.04.01,Ativo Total,Ativo Não Circulante,Intangível,Intangíveis,406628.0


In [ ]:
keys = [cd_conta_joincol[ind] for ind in ind_min_len]
key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

Join:

In [ ]:
ds_conta_4 = [df['DS_CONTA'][key_idx_dict[key]]
              for key in key_idx_dict
              for row in cd_conta_joincol
              if ".".join(row) == key]

In [ ]:
ds_conta_4 = [df['DS_CONTA'][key_idx_dict[key]]
              for row in cd_conta_split
              for key in key_idx_dict
              if ".".join(row[:4]) == key]

Insert:

In [ ]:
df.insert(5, column='DS_CONTA_4', value=ds_conta_4)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
2,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
3,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
4,ÚLTIMO,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
...,...,...,...,...,...,...,...,...
60,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
61,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
62,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
63,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Drop rows</h4>

Select rows with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
5,ÚLTIMO,1.01.02.03,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
6,ÚLTIMO,1.01.03.01,Ativo Total,Ativo Circulante,Contas a Receber,Clientes,Clientes,1239315.0
...,...,...,...,...,...,...,...,...
46,ÚLTIMO,1.02.03.01,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Operação,Imobilizado em Operação,3377237.0
47,ÚLTIMO,1.02.03.02,Ativo Total,Ativo Não Circulante,Imobilizado,Direito de Uso em Arrendamento,Direito de Uso em Arrendamento,0.0
48,ÚLTIMO,1.02.03.03,Ativo Total,Ativo Não Circulante,Imobilizado,Imobilizado em Andamento,Imobilizado em Andamento,135404.0
49,ÚLTIMO,1.02.04.01,Ativo Total,Ativo Não Circulante,Intangível,Intangíveis,Intangíveis,406628.0


For each key which was used to join, check if there is at least another row in `CD_CONTA` which code starts with the same key (example: `CD_CONTA` '1.01' starting with '1'; `CD_CONTA` '1.01.01' starting with '1.01', etc.)

In [ ]:
for ind in ind_min_len:
    mask = [True]*len(ds_conta_4)
    mask[ind] = False
    if any(row[:min_len] == cd_conta_joincol[ind] for row in cd_conta_split[mask]):
        df = df.drop(ind)
    else:
        df.loc[ind, 'CD_CONTA'] = df.loc[ind, 'CD_CONTA'] + '.00'

df.reset_index(inplace=True, drop=True)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,ÚLTIMO,1.01.02.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.03.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...,...
49,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


## <h3>ROUND 5</h3>

In [ ]:
round=5

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,ÚLTIMO,1.01.02.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.03.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...,...
49,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


### <h4>Join</h4>

In [ ]:
cd_conta_split = pd.Series([string.split('.') for string in df['CD_CONTA']])
cd_conta_joincol = [row[:5] for row in cd_conta_split]
# cd_conta_joincol: column with keys used to join
# row[:5] round = 5

Select rows with minimum length

In [ ]:
len_key = [len(row) for row in cd_conta_split]  # len(row) = number of elements separated by '.' in CD_CONTA
min_len = np.array(len_key).min()  # min_len = minimum length
ind_min_len = np.where(len_key==min_len)[0]  # ind_min_len = index of key 'CD_CONTA' with minimum length

In [ ]:
df.iloc[ind_min_len]

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA,VL_CONTA
0,ÚLTIMO,1.01.01.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,ÚLTIMO,1.01.02.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.03.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...,...
49,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0


In [ ]:
keys = [cd_conta_joincol[ind] for ind in ind_min_len]
key_idx_dict = {".".join(key): ind for key, ind in zip(keys, ind_min_len)}

Join:

In [ ]:
ds_conta_5 = [df['DS_CONTA'][key_idx_dict[key]]
              for key in key_idx_dict
              for row in cd_conta_joincol
              if ".".join(row) == key]

In [ ]:
ds_conta_5 = [df['DS_CONTA'][key_idx_dict[key]]
              for row in cd_conta_split
              for key in key_idx_dict
              if ".".join(row[:5]) == key]

Insert:

In [ ]:
if pd.Series(ds_conta_5).equals(df['DS_CONTA']):
    df.rename(columns={'DS_CONTA': 'DS_CONTA_5'}, inplace=True)
    df.to_csv("output.csv", index=False)
    exit
else:
    df.insert(6, column='DS_CONTA_5', value=ds_conta_5)

In [ ]:
df

,ORDEM_EXERC,CD_CONTA,DS_CONTA_1,DS_CONTA_2,DS_CONTA_3,DS_CONTA_4,DS_CONTA_5,VL_CONTA
0,ÚLTIMO,1.01.01.00.00,Ativo Total,Ativo Circulante,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,Caixa e Equivalentes de Caixa,1728413.0
1,ÚLTIMO,1.01.02.01.01,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos para Negociação,0.0
2,ÚLTIMO,1.01.02.01.02,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Títulos Designados a Valor Justo,0.0
3,ÚLTIMO,1.01.02.02.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas a Valor Justo...,Aplicações Financeiras Avaliadas a Valor Justo...,0.0
4,ÚLTIMO,1.01.02.03.00,Ativo Total,Ativo Circulante,Aplicações Financeiras,Aplicações Financeiras Avaliadas ao Custo Amor...,Aplicações Financeiras Avaliadas ao Custo Amor...,0.0
...,...,...,...,...,...,...,...,...
49,ÚLTIMO,1.02.04.02.07,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Caetex Florestal,8767.0
50,ÚLTIMO,1.02.04.02.08,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cerâmica Urussanga em...,92944.0
51,ÚLTIMO,1.02.04.02.09,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Massima Revestimentos...,6110.0
52,ÚLTIMO,1.02.04.02.10,Ativo Total,Ativo Não Circulante,Intangível,Goodwill,Goodwill na aquisição da Cecrisa Revestimentos...,168430.0
